# EDGAR Credit Clustering Starter

This notebook pivots the EDGAR fundamentals database into issuer-year credit features and runs **unsupervised clustering**.

Core framing:

- No credit labels at this stage.
- Cluster companies inside comparable sectors/accounting regimes.
- Start with either:
  - `major_sector`: SIC major divisions from the first notebook, or
  - `financial_flag`: Financial vs Non-financial.
- Target: **5 clusters per selected sector group**, then interpret clusters using medians and representative tickers.

In [1]:
import os
import glob
import duckdb
import numpy as np
import pandas as pd

from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

## 1. Connect to DuckDB

Expected local inputs from the previous notebook:

- `financials.duckdb`
- `raw_financial_facts_parquet/*.parquet`
- optionally `03_fundamental_universe_ticker_sic_industry.csv`

In [2]:
DB_PATH = "financials.duckdb"
PARQUET_GLOB = "raw_financial_facts_parquet/*.parquet"
UNIVERSE_CSV = "03_fundamental_universe_ticker_sic_industry.csv"

con = duckdb.connect(DB_PATH)

if glob.glob(PARQUET_GLOB):
    con.execute(f"""
    CREATE OR REPLACE VIEW raw_facts AS
    SELECT *
    FROM read_parquet('{PARQUET_GLOB}')
    """)
else:
    print("No parquet files found. Assuming raw_facts already exists in DuckDB.")

schema = con.execute("DESCRIBE raw_facts").df()
schema

,column_name,column_type,null,key,default,extra
0,concept,VARCHAR,YES,None,None,None
1,label,VARCHAR,YES,None,None,None
2,value,DOUBLE,YES,None,None,None
3,numeric_value,DOUBLE,YES,None,None,None
4,unit,VARCHAR,YES,None,None,None
5,period_type,VARCHAR,YES,None,None,None
6,period_start,VARCHAR,YES,None,None,None
7,period_end,VARCHAR,YES,None,None,None
8,fiscal_year,BIGINT,YES,None,None,None
9,fiscal_period,VARCHAR,YES,None,None,None


In [3]:
summary = con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT ticker) AS tickers,
    COUNT(DISTINCT concept) AS concepts,
    MIN(fiscal_year) AS min_year,
    MAX(fiscal_year) AS max_year
FROM raw_facts
""").df()
summary

,rows,tickers,concepts,min_year,max_year
0,100149899,8194,13256,0,44012


## 2. Sector mapping

The first notebook used SIC major divisions. For clustering, we keep that, but also create a simpler `financial_flag` because financial companies need separate treatment.

In [4]:
def map_sic_major_division(sic):
    if pd.isna(sic):
        return "Unknown"
    try:
        sic = int(float(sic))
    except Exception:
        return "Unknown"

    if 100 <= sic < 1000:
        return "Agriculture"
    if 1000 <= sic < 1500:
        return "Mining / Energy"
    if 1500 <= sic < 1800:
        return "Construction"
    if 2000 <= sic < 4000:
        return "Manufacturing / Industrials"
    if 4000 <= sic < 5000:
        return "Transportation / Utilities"
    if 5000 <= sic < 6000:
        return "Wholesale / Retail"
    if 6000 <= sic < 6800:
        return "Finance / Insurance / Real Estate"
    if 7000 <= sic < 9000:
        return "Services"
    if 9100 <= sic < 9730:
        return "Public Administration"
    return "Other"

def map_financial_flag(sic):
    if pd.isna(sic):
        return "Unknown"
    try:
        sic = int(float(sic))
    except Exception:
        return "Unknown"
    return "Financial" if 6000 <= sic < 6800 else "Non-financial"

In [5]:
# Optional enriched universe from the first notebook.
if os.path.exists(UNIVERSE_CSV):
    universe = pd.read_csv(UNIVERSE_CSV)
    universe["major_sector"] = universe.get("industry", universe["sic"].apply(map_sic_major_division))
    universe["financial_flag"] = universe["sic"].apply(map_financial_flag)
    universe = universe[[c for c in ["ticker", "cik", "name", "sic", "Industry Title", "major_sector", "financial_flag"] if c in universe.columns]].drop_duplicates()
else:
    universe = con.execute("""
    SELECT DISTINCT ticker, cik, company_name AS name, sic
    FROM raw_facts
    WHERE ticker IS NOT NULL
    """).df()
    universe["major_sector"] = universe["sic"].apply(map_sic_major_division)
    universe["financial_flag"] = universe["sic"].apply(map_financial_flag)

universe["major_sector"].value_counts(dropna=False).to_frame("companies")

,companies
major_sector,
Manufacturing / Industrials,2878
Finance / Insurance / Real Estate,1979
Services,1557
Transportation / Utilities,704
Wholesale / Retail,484
Mining / Energy,463
Construction,98
Agriculture,52


In [6]:
universe["financial_flag"].value_counts(dropna=False).to_frame("companies")

,companies
financial_flag,
Non-financial,6236
Financial,1979


## 3. Concept map

EDGAR concepts are noisy. We map multiple possible GAAP concepts into canonical feature names. The query below keeps only target concepts, then aggregates issuer-year-canonical-feature values.

For the first pass, we use **FY periods only** and clean fiscal years to 2020–2025.

In [7]:
CONCEPT_MAP = {
    # Balance sheet
    "assets": ["us-gaap:Assets"],
    "assets_current": ["us-gaap:AssetsCurrent"],
    "liabilities": ["us-gaap:Liabilities"],
    "liabilities_current": ["us-gaap:LiabilitiesCurrent"],
    "cash": [
        "us-gaap:CashAndCashEquivalentsAtCarryingValue",
        "us-gaap:CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents",
        "us-gaap:CashAndDueFromBanks",
    ],
    "receivables": ["us-gaap:AccountsReceivableNetCurrent"],
    "inventory": ["us-gaap:InventoryNet"],
    "ppe": ["us-gaap:PropertyPlantAndEquipmentNet"],
    "goodwill": ["us-gaap:Goodwill"],
    "equity": [
        "us-gaap:StockholdersEquity",
        "us-gaap:StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
    ],
    "long_term_debt": ["us-gaap:LongTermDebt", "us-gaap:LongTermDebtNoncurrent"],
    "short_term_debt": [
        "us-gaap:ShortTermBorrowings",
        "us-gaap:LongTermDebtCurrent",
        "us-gaap:CurrentPortionOfLongTermDebt",
    ],

    # Income statement
    "revenue": [
        "us-gaap:Revenues",
        "us-gaap:SalesRevenueNet",
        "us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax",
    ],
    "gross_profit": ["us-gaap:GrossProfit"],
    "operating_income": ["us-gaap:OperatingIncomeLoss"],
    "net_income": ["us-gaap:NetIncomeLoss", "us-gaap:ProfitLoss"],
    "interest_expense": ["us-gaap:InterestExpense", "us-gaap:InterestExpenseNonOperating"],
    "sga": ["us-gaap:SellingGeneralAndAdministrativeExpense"],
    "rd": ["us-gaap:ResearchAndDevelopmentExpense"],

    # Cash flow
    "cfo": ["us-gaap:NetCashProvidedByUsedInOperatingActivities"],
    "capex": ["us-gaap:PaymentsToAcquirePropertyPlantAndEquipment"],
}

concept_lookup = []
for canonical, concepts in CONCEPT_MAP.items():
    for concept in concepts:
        concept_lookup.append({"canonical_feature": canonical, "concept": concept})
concept_lookup = pd.DataFrame(concept_lookup)
concept_lookup.head(20)

,canonical_feature,concept
0,assets,us-gaap:Assets
1,assets_current,us-gaap:AssetsCurrent
2,liabilities,us-gaap:Liabilities
3,liabilities_current,us-gaap:LiabilitiesCurrent
4,cash,us-gaap:CashAndCashEquivalentsAtCarryingValue
5,cash,us-gaap:CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents
6,cash,us-gaap:CashAndDueFromBanks
7,receivables,us-gaap:AccountsReceivableNetCurrent
8,inventory,us-gaap:InventoryNet
9,ppe,us-gaap:PropertyPlantAndEquipmentNet


In [8]:
con.register("concept_lookup", concept_lookup)

# Detect the numeric column used in your raw facts export.
cols = set(schema["column_name"].str.lower())
if "numeric_value" in cols:
    value_col = "numeric_value"
elif "value" in cols:
    value_col = "value"
else:
    raise ValueError("Could not find numeric_value or value column in raw_facts.")

# Pick a filing/period sort column if available. This improves deduplication.
schema_cols = schema["column_name"].tolist()
lower_to_actual = {c.lower(): c for c in schema_cols}
sort_candidates = ["filing_date", "filed", "period_end", "end", "start"]
sort_col = next((lower_to_actual[c] for c in sort_candidates if c in lower_to_actual), None)
print("value_col:", value_col, "| sort_col:", sort_col)

value_col: numeric_value | sort_col: period_end


In [10]:
# We aggregate by median to reduce duplicate filing/noisy restatement impact.
# Later: replace this with a stricter latest-filing selection if the raw columns support it.
con.execute(f"""
CREATE OR REPLACE TABLE issuer_year_facts AS
SELECT
    rf.ticker,
    TRY_CAST(rf.cik AS VARCHAR) AS cik,
    ANY_VALUE(rf.company_name) AS company_name,
    TRY_CAST(rf.sic AS INTEGER) AS sic,
    TRY_CAST(rf.fiscal_year AS INTEGER) AS fiscal_year,
    cl.canonical_feature,
    MEDIAN(TRY_CAST(rf.{value_col} AS DOUBLE)) AS value
FROM raw_facts rf
JOIN concept_lookup cl
    ON rf.concept = cl.concept
WHERE TRY_CAST(rf.fiscal_year AS INTEGER) BETWEEN 2020 AND 2025
  AND rf.fiscal_period = 'FY'
  AND TRY_CAST(rf.{value_col} AS DOUBLE) IS NOT NULL
GROUP BY 1,2,4,5,6
""")

facts_summary = con.execute("""
SELECT
    canonical_feature,
    COUNT(*) AS row_count,
    COUNT(DISTINCT ticker) AS ticker_count
FROM issuer_year_facts
GROUP BY 1
ORDER BY ticker_count DESC
""").df()

facts_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,canonical_feature,row_count,ticker_count
0,net_income,36055,7171
1,assets,36044,7147
2,cfo,35527,7106
3,equity,35594,7048
4,cash,34942,7035
5,liabilities,31638,6462
6,revenue,27296,5841
7,operating_income,27902,5778
8,ppe,26720,5771
9,liabilities_current,27656,5673


## 4. Build issuer-year panel

One row per ticker-year. Then attach sector labels.

In [11]:
panel = con.execute("""
PIVOT issuer_year_facts
ON canonical_feature
USING MAX(value)
GROUP BY ticker, cik, company_name, sic, fiscal_year
""").df()

panel["major_sector"] = panel["sic"].apply(map_sic_major_division)
panel["financial_flag"] = panel["sic"].apply(map_financial_flag)

panel.shape, panel.head()

((36288, 28),
   ticker      cik                                      company_name   sic  \
 0     FE  1031296                                  FIRSTENERGY CORP  4911   
 1  FNMFO   310522  FEDERAL NATIONAL MORTGAGE ASSOCIATION FANNIE MAE  6111   
 2   MYFW  1327607                       First Western Financial Inc  6022   
 3   MYCB  1556801                            My City Builders, Inc.  6500   
 4   AMZN  1018724                                    AMAZON COM INC  5961   
 
    fiscal_year        assets  assets_current      capex          cash  \
 0         2025  5.397400e+10    2.877500e+09        NaN  1.325000e+08   
 1         2021  4.107458e+12             NaN        NaN  5.541500e+10   
 2         2023  2.921105e+09             NaN  2657000.0  1.965120e+08   
 3         2024  2.859544e+06    1.027060e+06   907588.5  2.024500e+04   
 4         2025  6.248940e+11    2.099750e+11        NaN  8.054550e+10   
 
             cfo        equity      goodwill  gross_profit  interest_e

In [ ]:
panel.groupby(["major_sector", "fiscal_year"]).size().unstack(fill_value=0)

## 5. Credit feature engineering

The feature set has two layers:

1. Universal features usable for most companies.
2. Non-financial-only features where working capital, debt, capex, and interest coverage are more meaningful.

Financials should not be forced into industrial ratios like current ratio or debt/assets without sector-specific accounting logic.

In [12]:
def safe_div(n, d):
    n = pd.to_numeric(n, errors="coerce")
    d = pd.to_numeric(d, errors="coerce")
    out = n / d.replace({0: np.nan})
    return out.replace([np.inf, -np.inf], np.nan)

for c in CONCEPT_MAP.keys():
    if c not in panel.columns:
        panel[c] = np.nan

panel["total_debt"] = panel["long_term_debt"].fillna(0) + panel["short_term_debt"].fillna(0)
panel.loc[panel[["long_term_debt", "short_term_debt"]].isna().all(axis=1), "total_debt"] = np.nan
panel["fcf"] = panel["cfo"] - panel["capex"].abs()

# Universal-ish scale and solvency ratios
panel["log_assets"] = np.log1p(panel["assets"].clip(lower=0))
panel["liabilities_to_assets"] = safe_div(panel["liabilities"], panel["assets"])
panel["equity_to_assets"] = safe_div(panel["equity"], panel["assets"])
panel["cash_to_assets"] = safe_div(panel["cash"], panel["assets"])
panel["revenue_to_assets"] = safe_div(panel["revenue"], panel["assets"])
panel["net_income_to_assets"] = safe_div(panel["net_income"], panel["assets"])
panel["cfo_to_assets"] = safe_div(panel["cfo"], panel["assets"])

# Industrial / non-financial ratios
panel["debt_to_assets"] = safe_div(panel["total_debt"], panel["assets"])
panel["debt_to_equity"] = safe_div(panel["total_debt"], panel["equity"])
panel["current_ratio"] = safe_div(panel["assets_current"], panel["liabilities_current"])
panel["quick_ratio"] = safe_div(panel["cash"].fillna(0) + panel["receivables"].fillna(0), panel["liabilities_current"])
panel.loc[panel[["cash", "receivables"]].isna().all(axis=1), "quick_ratio"] = np.nan
panel["gross_margin"] = safe_div(panel["gross_profit"], panel["revenue"])
panel["operating_margin"] = safe_div(panel["operating_income"], panel["revenue"])
panel["interest_coverage"] = safe_div(panel["operating_income"], panel["interest_expense"].abs())
panel["cfo_to_liabilities"] = safe_div(panel["cfo"], panel["liabilities"])
panel["fcf_to_debt"] = safe_div(panel["fcf"], panel["total_debt"])
panel["capex_to_revenue"] = safe_div(panel["capex"].abs(), panel["revenue"])

# Winsorize extreme ratios before clustering.
ratio_cols = [
    "liabilities_to_assets", "equity_to_assets", "cash_to_assets", "revenue_to_assets",
    "net_income_to_assets", "cfo_to_assets", "debt_to_assets", "debt_to_equity",
    "current_ratio", "quick_ratio", "gross_margin", "operating_margin", "interest_coverage",
    "cfo_to_liabilities", "fcf_to_debt", "capex_to_revenue"
]
for c in ratio_cols:
    lo, hi = panel[c].quantile([0.01, 0.99])
    panel[c] = panel[c].clip(lo, hi)

panel[["ticker", "fiscal_year", "major_sector", "financial_flag", "log_assets"] + ratio_cols].head()

,ticker,fiscal_year,major_sector,financial_flag,log_assets,liabilities_to_assets,equity_to_assets,cash_to_assets,revenue_to_assets,net_income_to_assets,cfo_to_assets,debt_to_assets,debt_to_equity,current_ratio,quick_ratio,gross_margin,operating_margin,interest_coverage,cfo_to_liabilities,fcf_to_debt,capex_to_revenue
0,FE,2025,Transportation / Utilities,Non-financial,24.711768,0.743895,0.231269,0.002455,0.249602,0.020649,0.053563,0.011793,0.050991,0.560479,0.342910,NaN,0.168201,NaN,0.072003,NaN,NaN
1,FNMFO,2021,Finance / Insurance / Real Estate,Financial,29.043825,0.991160,0.006150,0.013491,NaN,0.003447,-0.001157,0.983450,88.340478,NaN,NaN,NaN,NaN,NaN,-0.001168,NaN,NaN
2,MYFW,2023,Finance / Insurance / Real Estate,Financial,21.795228,0.917223,0.082456,0.067273,0.032630,0.004608,0.012009,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.013093,NaN,0.027876
3,MYCB,2024,Finance / Insurance / Real Estate,Financial,14.866173,1.096587,0.107906,0.007080,0.018969,-0.349152,-0.129462,0.481159,4.459041,0.371063,0.007314,NaN,-3.978772,-22.389387,-0.118059,-0.928697,8.006904
4,AMZN,2025,Wholesale / Retail,Non-financial,27.160848,NaN,0.390342,0.128895,1.020908,0.094813,0.185435,0.101498,0.260023,1.056648,0.715262,NaN,0.107519,NaN,NaN,NaN,NaN


In [13]:
feature_coverage = (
    panel[["financial_flag", "major_sector"] + ["log_assets"] + ratio_cols]
    .groupby(["financial_flag", "major_sector"])
    .agg(lambda s: s.notna().mean())
)
feature_coverage.round(2)

log_assets  \
financial_flag major_sector                                    
Financial      Finance / Insurance / Real Estate        0.99   
Non-financial  Agriculture                              0.99   
               Construction                             1.00   
               Manufacturing / Industrials              1.00   
               Mining / Energy                          0.98   
               Services                                 0.99   
               Transportation / Utilities               1.00   
               Wholesale / Retail                       0.99   

                                                  liabilities_to_assets  \
financial_flag major_sector                                               
Financial      Finance / Insurance / Real Estate                   0.98   
Non-financial  Agriculture                                         0.83   
               Construction                                        0.85   
               Manufacturing / Industrials                         0.85   
               Mining / Energy                                     0.80   
               Services                                            0.90   
               Transportation / Utilities                          0.61   
               Wholesale / Retail                                  0.78   

                                                  equity_to_assets  \
financial_flag major_sector                                          
Financial      Finance / Insurance / Real Estate              0.96   
Non-financial  Agriculture                                    0.95   
               Construction                                   0.98   
               Manufacturing / Industrials                    0.99   
               Mining / Energy                                0.94   
               Services                                       0.98   
               Transportation / Utilities                     0.94   
               Wholesale / Retail                             0.97   

                                                  cash_to_assets  \
financial_flag major_sector                                        
Financial      Finance / Insurance / Real Estate            0.95   
Non-financial  Agriculture                                  0.96   
               Construction                                 0.93   
               Manufacturing / Industrials                  0.97   
               Mining / Energy                              0.91   
               Services                                     0.95   
               Transportation / Utilities                   0.98   
               Wholesale / Retail                           0.93   

                                                  revenue_to_assets  \
financial_flag major_sector                                           
Financial      Finance / Insurance / Real Estate               0.60   
Non-financial  Agriculture                                     0.89   
               Construction                                    0.87   
               Manufacturing / Industrials                     0.76   
               Mining / Energy                                 0.66   
               Services                                        0.86   
               Transportation / Utilities                      0.85   
               Wholesale / Retail                              0.89   

                                                  net_income_to_assets  \
financial_flag major_sector                                              
Financial      Finance / Insurance / Real Estate                  0.98   
Non-financial  Agriculture                                        0.97   
               Construction                                       0.97   
               Manufacturing / Industrials                        0.99   
               Mining / Energy                                    0.96   
               Services                        

## 6. Decide clustering segmentation

Default recommendation:

- First run `financial_flag`: Financial vs Non-financial.
- Then run `major_sector` for non-financial sectors with enough companies.
- Keep 5 clusters only where the group has enough observations. A practical floor is 100 issuer-years, preferably 200+.

In [14]:
SEGMENT_COL = "financial_flag"   # alternatives: "major_sector"
N_CLUSTERS = 5
MIN_ROWS_PER_SEGMENT = 100

UNIVERSAL_FEATURES = [
    "log_assets",
    "liabilities_to_assets",
    "equity_to_assets",
    "cash_to_assets",
    "revenue_to_assets",
    "net_income_to_assets",
    "cfo_to_assets",
]

NONFIN_FEATURES = UNIVERSAL_FEATURES + [
    "debt_to_assets",
    "debt_to_equity",
    "current_ratio",
    "quick_ratio",
    "gross_margin",
    "operating_margin",
    "interest_coverage",
    "cfo_to_liabilities",
    "fcf_to_debt",
    "capex_to_revenue",
]

FIN_FEATURES = [
    "log_assets",
    "liabilities_to_assets",
    "equity_to_assets",
    "cash_to_assets",
    "revenue_to_assets",
    "net_income_to_assets",
    "cfo_to_assets",
]

FEATURES_BY_FIN_FLAG = {
    "Financial": FIN_FEATURES,
    "Non-financial": NONFIN_FEATURES,
    "Unknown": UNIVERSAL_FEATURES,
}

In [15]:
def get_features_for_segment(segment_name, segment_col):
    if segment_col == "financial_flag":
        return FEATURES_BY_FIN_FLAG.get(segment_name, UNIVERSAL_FEATURES)
    # For major sectors, treat Finance / Insurance / Real Estate with financial features.
    if segment_name == "Finance / Insurance / Real Estate":
        return FIN_FEATURES
    return NONFIN_FEATURES


def cluster_segment(df, segment_name, segment_col, n_clusters=5, min_rows=100):
    features = get_features_for_segment(segment_name, segment_col)
    use = df[df[segment_col] == segment_name].copy()
    use = use.dropna(subset=["ticker", "fiscal_year"])

    # Keep features with at least 50% availability inside this segment.
    availability = use[features].notna().mean().sort_values(ascending=False)
    features = availability[availability >= 0.50].index.tolist()

    if len(use) < min_rows or len(features) < 4:
        return None, {"segment": segment_name, "status": "skipped", "rows": len(use), "features": len(features)}

    X = use[features]
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
        ("cluster", KMeans(n_clusters=n_clusters, n_init=50, random_state=42)),
    ])
    labels = pipe.fit_predict(X)
    use["cluster"] = labels
    use["cluster_key"] = use[segment_col].astype(str) + "__" + use["cluster"].astype(str)

    Xt = pipe[:-1].transform(X)
    metrics = {
        "segment": segment_name,
        "status": "clustered",
        "rows": len(use),
        "features": len(features),
        "feature_list": features,
        "silhouette": silhouette_score(Xt, labels) if len(set(labels)) > 1 else np.nan,
        "calinski_harabasz": calinski_harabasz_score(Xt, labels) if len(set(labels)) > 1 else np.nan,
        "davies_bouldin": davies_bouldin_score(Xt, labels) if len(set(labels)) > 1 else np.nan,
    }
    return use, metrics

clustered_parts = []
metrics = []
for segment_name in sorted(panel[SEGMENT_COL].dropna().unique()):
    clustered, m = cluster_segment(panel, segment_name, SEGMENT_COL, N_CLUSTERS, MIN_ROWS_PER_SEGMENT)
    metrics.append(m)
    if clustered is not None:
        clustered_parts.append(clustered)

metrics_df = pd.DataFrame(metrics)
metrics_df

,segment,status,rows,features,feature_list,silhouette,calinski_harabasz,davies_bouldin
0,Financial,clustered,9483,7,"[log_assets, net_income_to_assets, liabilities_to_assets, cfo_to_assets, equity_to_assets, cash_to_assets, revenue_t...",0.894714,53215.465566,0.604899
1,Non-financial,clustered,26805,14,"[log_assets, net_income_to_assets, equity_to_assets, cfo_to_assets, current_ratio, cash_to_assets, quick_ratio, liab...",0.893684,77718.483790,0.624143


In [17]:
clustered_panel = pd.concat(clustered_parts, ignore_index=True) if clustered_parts else pd.DataFrame()
clustered_panel.shape

(36288, 49)

## 7. Interpret clusters

Clusters only become useful after interpretation. We rank them by stress indicators:

- higher liabilities/assets and debt/assets = worse
- lower equity/assets, cash/assets, coverage, CFO/assets = worse
- lower profitability = worse

This ranking is not a credit rating. It is a practical credit-risk ordering for analyst review.

In [18]:
INTERPRET_FEATURES = [
    "log_assets", "liabilities_to_assets", "equity_to_assets", "cash_to_assets",
    "net_income_to_assets", "cfo_to_assets", "debt_to_assets", "current_ratio",
    "interest_coverage", "cfo_to_liabilities", "fcf_to_debt", "operating_margin"
]
INTERPRET_FEATURES = [c for c in INTERPRET_FEATURES if c in clustered_panel.columns]

cluster_profile = (
    clustered_panel
    .groupby([SEGMENT_COL, "cluster"])
    .agg(
        issuer_years=("ticker", "size"),
        issuers=("ticker", "nunique"),
        **{f"median_{c}": (c, "median") for c in INTERPRET_FEATURES}
    )
    .reset_index()
)
cluster_profile

,financial_flag,cluster,issuer_years,issuers,median_log_assets,median_liabilities_to_assets,median_equity_to_assets,median_cash_to_assets,median_net_income_to_assets,median_cfo_to_assets,median_debt_to_assets,median_current_ratio,median_interest_coverage,median_cfo_to_liabilities,median_fcf_to_debt,median_operating_margin
0,Financial,0,9269,1771,22.409511,0.826783,0.141905,0.036018,0.008557,0.014671,0.107872,1.633342,1.197544,0.016928,0.242601,0.110054
1,Financial,1,34,20,9.888486,32.638122,-29.823595,0.294639,-17.663002,-5.716679,1.188748,0.008271,-23.489127,-0.200884,-4.929024,-8.825884
2,Financial,2,38,30,13.879634,1.861857,-0.747373,0.125987,-3.716717,-1.040360,0.080065,0.328348,-71.244088,-0.425235,NaN,-3.243377
3,Financial,3,6,6,10.087266,32.638122,-30.214917,0.154238,-11.710699,-1.745552,NaN,0.019554,-22.539186,-0.267042,NaN,-6.325416
4,Financial,4,136,74,15.752314,0.686235,0.133294,0.126254,-1.333260,-0.500493,0.219667,0.921252,-10.173030,-0.452910,-4.523888,-4.190583
5,Non-financial,0,25830,5385,19.939457,0.526677,0.365342,0.108594,-0.006998,0.023862,0.248427,1.762079,0.661597,-0.001222,0.066093,0.018447
6,Non-financial,1,254,162,16.006941,0.536367,0.203121,0.258942,-1.061760,-0.680736,0.099988,1.704461,-33.391866,-1.048584,-5.089333,-448.124301
7,Non-financial,2,150,116,16.937800,0.456755,0.285010,0.226900,-0.890274,-0.583184,0.213370,2.992784,-20.456543,-1.008894,-2.815689,-217.458047
8,Non-financial,3,491,314,16.978026,0.430711,0.335377,0.299409,-0.758572,-0.564858,0.127686,3.159413,-25.109190,-1.094030,-4.197120,-73.928715
9,Non-financial,4,80,66,21.733775,0.970293,0.009188,0.066618,-0.017711,0.047446,0.557305,1.193932,2.955483,0.044419,0.027424,0.012422


In [21]:
cluster_size_check = (
    clustered_panel
    .groupby([SEGMENT_COL, "cluster"])
    .size()
    .reset_index(name="row_count")
    .sort_values([SEGMENT_COL, "row_count"], ascending=[True, False])
)

cluster_size_check

,financial_flag,cluster,row_count
0,Financial,0,9269
4,Financial,4,136
2,Financial,2,38
1,Financial,1,34
3,Financial,3,6
5,Non-financial,0,25830
8,Non-financial,3,491
6,Non-financial,1,254
7,Non-financial,2,150
9,Non-financial,4,80


In [22]:
profile_cols = [
    c for c in [
        "log_assets",
        "liabilities_to_assets",
        "debt_to_assets",
        "equity_to_assets",
        "current_ratio",
        "quick_ratio",
        "cash_to_assets",
        "net_income_to_assets",
        "cfo_to_assets",
        "revenue_to_assets",
        "interest_coverage",
        "fcf_to_debt",
        "operating_margin",
        "gross_margin",
    ]
    if c in clustered_panel.columns
]

cluster_medians = (
    clustered_panel
    .groupby([SEGMENT_COL, "cluster"])[profile_cols]
    .median()
    .round(3)
)

cluster_medians

log_assets  liabilities_to_assets  debt_to_assets  \
financial_flag cluster                                                      
Financial      0            22.410                  0.827           0.108   
               1             9.888                 32.638           1.189   
               2            13.880                  1.862           0.080   
               3            10.087                 32.638             NaN   
               4            15.752                  0.686           0.220   
Non-financial  0            19.939                  0.527           0.248   
               1            16.007                  0.536           0.100   
               2            16.938                  0.457           0.213   
               3            16.978                  0.431           0.128   
               4            21.734                  0.970           0.557   

                        equity_to_assets  current_ratio  quick_ratio  \
financial_flag cluster                                                 
Financial      0                   0.142          1.633        0.717   
               1                 -29.824          0.008        0.006   
               2                  -0.747          0.328        0.077   
               3                 -30.215          0.020        0.005   
               4                   0.133          0.921        0.263   
Non-financial  0                   0.365          1.762        0.886   
               1                   0.203          1.704        0.981   
               2                   0.285          2.993        1.204   
               3                   0.335          3.159        1.447   
               4                   0.009          1.194        0.465   

                        cash_to_assets  net_income_to_assets  cfo_to_assets  \
financial_flag cluster                                                        
Financial      0                 0.036                 0.009          0.015   
               1                 0.295               -17.663         -5.717   
               2                 0.126                -3.717         -1.040   
               3                 0.154               -11.711         -1.746   
               4                 0.126                -1.333         -0.500   
Non-financial  0                 0.109                -0.007          0.024   
               1                 0.259                -1.062         -0.681   
               2                 0.227                -0.890         -0.583   
               3                 0.299                -0.759         -0.565   
               4                 0.067                -0.018          0.047   

                        revenue_to_assets  interest_coverage  fcf_to_debt  \
financial_flag cluster                                                      
Financial      0                    0.099              1.198        0.243   
               1                    0.461            -23.489       -4.929   
               2                    0.440            -71.244          NaN   
               3                    0.630            -22.539          NaN   
               4                    0.209            -10.173       -4.524   
Non-financial  0                    0.486              0.662        0.066   
               1                    0.001            -33.392       -5.089   
               2                    0.004            -20.457       -2.816   
               3                    0.011            -25.109       -4.197   
               4                    0.534              2.955        0.027   

                        operating_margin  gross_margin  
financial_flag cluster                                  
Financial      0                   0.110         0.406  
               1                  -8.826         0.614  
               2                  -3.243         0.254  
               3                  -6.325         0.568  
               4  

In [23]:
feature_extremes = (
    clustered_panel[profile_cols]
    .quantile([0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999])
    .T
    .round(3)
)

feature_extremes

,0.001,0.010,0.050,0.500,0.950,0.990,0.999
log_assets,0.000,9.370,13.911,20.624,25.505,28.766,31.391
liabilities_to_assets,0.009,0.009,0.070,0.603,2.223,32.625,32.638
debt_to_assets,0.000,0.000,0.005,0.222,0.846,1.676,1.676
equity_to_assets,-30.727,-30.663,-1.173,0.312,0.874,1.102,1.102
current_ratio,0.006,0.006,0.114,1.750,12.336,29.847,29.869
quick_ratio,0.000,0.000,0.020,0.877,6.470,16.758,16.759
cash_to_assets,0.000,0.000,0.003,0.082,0.697,1.018,1.019
net_income_to_assets,-17.663,-17.663,-1.836,0.003,0.124,0.262,0.262
cfo_to_assets,-5.717,-5.700,-1.085,0.015,0.174,0.300,0.300
revenue_to_assets,0.000,0.000,0.006,0.358,1.898,3.897,3.905


In [24]:
industry_cluster_mix = (
    clustered_panel
    .groupby([SEGMENT_COL, "cluster", "major_sector"])
    .size()
    .reset_index(name="row_count")
)

industry_cluster_mix["cluster_total"] = (
    industry_cluster_mix
    .groupby([SEGMENT_COL, "cluster"])["row_count"]
    .transform("sum")
)

industry_cluster_mix["pct_of_cluster"] = (
    industry_cluster_mix["row_count"] / industry_cluster_mix["cluster_total"]
)

industry_cluster_mix = (
    industry_cluster_mix
    .sort_values([SEGMENT_COL, "cluster", "pct_of_cluster"], ascending=[True, True, False])
)

industry_cluster_mix.head(50)

,financial_flag,cluster,major_sector,row_count,cluster_total,pct_of_cluster
0,Financial,0,Finance / Insurance / Real Estate,9269,9269,1.000000
1,Financial,1,Finance / Insurance / Real Estate,34,34,1.000000
2,Financial,2,Finance / Insurance / Real Estate,38,38,1.000000
3,Financial,3,Finance / Insurance / Real Estate,6,6,1.000000
4,Financial,4,Finance / Insurance / Real Estate,136,136,1.000000
7,Non-financial,0,Manufacturing / Industrials,12455,25830,0.482191
9,Non-financial,0,Services,6407,25830,0.248045
10,Non-financial,0,Transportation / Utilities,2735,25830,0.105885
11,Non-financial,0,Wholesale / Retail,2229,25830,0.086295
8,Non-financial,0,Mining / Energy,1389,25830,0.053775


In [20]:
def add_credit_stress_score(profile):
    p = profile.copy()
    score = 0

    # Bad when high
    for c in ["median_liabilities_to_assets", "median_debt_to_assets"]:
        if c in p.columns:
            score += p.groupby(SEGMENT_COL)[c].rank(pct=True)

    # Bad when low
    for c in [
        "median_equity_to_assets", "median_cash_to_assets", "median_net_income_to_assets",
        "median_cfo_to_assets", "median_current_ratio", "median_interest_coverage",
        "median_cfo_to_liabilities", "median_fcf_to_debt", "median_operating_margin"
    ]:
        if c in p.columns:
            score += 1 - p.groupby(SEGMENT_COL)[c].rank(pct=True)

    p["credit_stress_score"] = score
    p["stress_rank_within_segment"] = p.groupby(SEGMENT_COL)["credit_stress_score"].rank(ascending=False, method="dense").astype(int)
    return p.sort_values([SEGMENT_COL, "stress_rank_within_segment"])

cluster_profile_ranked = add_credit_stress_score(cluster_profile)
cluster_profile_ranked

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

In [ ]:
# Representative tickers closest to each cluster centroid approximation using median feature distance.
def representatives(clustered_panel, segment_col, max_names=10):
    rows = []
    for (seg, cl), g in clustered_panel.groupby([segment_col, "cluster"]):
        features = get_features_for_segment(seg, segment_col)
        features = [f for f in features if f in g.columns and g[f].notna().mean() >= 0.50]
        if not features:
            continue
        med = g[features].median(numeric_only=True)
        X = g[features].copy()
        X = X.fillna(med)
        dist = ((X - med) ** 2).sum(axis=1) ** 0.5
        reps = g.loc[dist.sort_values().index, ["ticker", "company_name", "fiscal_year"]].head(max_names)
        rows.append({
            segment_col: seg,
            "cluster": cl,
            "representative_tickers": ", ".join(reps["ticker"].astype(str).unique()[:max_names]),
            "sample_companies": " | ".join(reps["company_name"].dropna().astype(str).unique()[:5]),
        })
    return pd.DataFrame(rows)

cluster_representatives = representatives(clustered_panel, SEGMENT_COL)
cluster_profile_ranked.merge(cluster_representatives, on=[SEGMENT_COL, "cluster"], how="left")

## 8. Visual sanity check with PCA

This is not proof of cluster quality. It just helps see whether clusters are obviously broken.

In [ ]:
if not clustered_panel.empty:
    first_segment = clustered_panel[SEGMENT_COL].value_counts().index[0]
    g = clustered_panel[clustered_panel[SEGMENT_COL] == first_segment].copy()
    features = get_features_for_segment(first_segment, SEGMENT_COL)
    features = [f for f in features if f in g.columns and g[f].notna().mean() >= 0.50]

    prep = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ])
    Xp = prep.fit_transform(g[features])
    pca = PCA(n_components=2, random_state=42)
    pcs = pca.fit_transform(Xp)

    plt.figure(figsize=(8, 6))
    plt.scatter(pcs[:, 0], pcs[:, 1], c=g["cluster"], s=12, alpha=0.65)
    plt.title(f"PCA view of clusters: {first_segment}")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.show()

    print("Explained variance:", pca.explained_variance_ratio_)

## 9. Compare 5 clusters vs alternatives

We want 5 clusters for business interpretability, but we should still test whether the data screams for fewer/more. Use this as governance, not as an automatic decision.

In [ ]:
def evaluate_k_range(df, segment_name, segment_col, k_values=range(2, 9)):
    features = get_features_for_segment(segment_name, segment_col)
    use = df[df[segment_col] == segment_name].copy()
    availability = use[features].notna().mean().sort_values(ascending=False)
    features = availability[availability >= 0.50].index.tolist()
    if len(use) < 100 or len(features) < 4:
        return pd.DataFrame()

    prep = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ])
    X = prep.fit_transform(use[features])

    rows = []
    for k in k_values:
        labels = KMeans(n_clusters=k, n_init=50, random_state=42).fit_predict(X)
        rows.append({
            "segment": segment_name,
            "k": k,
            "silhouette": silhouette_score(X, labels),
            "calinski_harabasz": calinski_harabasz_score(X, labels),
            "davies_bouldin": davies_bouldin_score(X, labels),
        })
    return pd.DataFrame(rows)

k_tests = []
for segment_name in sorted(panel[SEGMENT_COL].dropna().unique()):
    kt = evaluate_k_range(panel, segment_name, SEGMENT_COL)
    if not kt.empty:
        k_tests.append(kt)

k_tests = pd.concat(k_tests, ignore_index=True) if k_tests else pd.DataFrame()
k_tests

## 10. Save outputs

These are the handoff files for review and downstream modeling.

In [ ]:
OUTPUT_DIR = "credit_clustering_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

panel.to_parquet(os.path.join(OUTPUT_DIR, "issuer_year_feature_panel.parquet"), index=False)
clustered_panel.to_parquet(os.path.join(OUTPUT_DIR, f"clustered_panel_by_{SEGMENT_COL}.parquet"), index=False)
cluster_profile_ranked.to_csv(os.path.join(OUTPUT_DIR, f"cluster_profile_by_{SEGMENT_COL}.csv"), index=False)
metrics_df.to_csv(os.path.join(OUTPUT_DIR, f"cluster_metrics_by_{SEGMENT_COL}.csv"), index=False)
k_tests.to_csv(os.path.join(OUTPUT_DIR, f"cluster_k_tests_by_{SEGMENT_COL}.csv"), index=False)

print("Saved outputs to", OUTPUT_DIR)

## Recommended next move

Run this twice:

1. `SEGMENT_COL = "financial_flag"`
2. `SEGMENT_COL = "major_sector"`

Then compare:

- cluster sizes,
- silhouette / Davies-Bouldin,
- interpretability of medians,
- whether financials behave better when isolated.

The expected outcome is likely:

- financials separate,
- non-financials clustered by major sector where sample size is large enough,
- small SIC groups merged into a broader non-financial bucket.